# Le futur est déjà, en grande partie, dans les cours · *The future is already, largely, in the price*

Notebook compagnon du chapitre **20. La croissance déjà anticipée : pourquoi elle est souvent dans les cours** — [lire l'article](https://nmlab.io/ressources/croissance-deja-anticipee-dans-les-cours).
Companion notebook to chapter **20. Growth Already Anticipated: Why It's Often Already in the Price** — [read the article](https://nmlab.io/en/ressources/growth-already-priced-in).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère à partir des **données de Robert Shiller (CAPE)**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from **Robert Shiller's data (CAPE)**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import urllib.request
import pandas as pd
from pandas import DataFrame

SHILLER_URL = "http://www.econ.yale.edu/~shiller/data/ie_data.xls"

def load_shiller() -> DataFrame:
    """Données historiques de Robert Shiller (ie_data.xls) : CAPE mensuel et rendement
    réel des actions sur 10 ans (colonnes précalculées par Shiller), depuis 1871.
    Robert Shiller's historical data: monthly CAPE and the 10-year real stock return."""
    try:                                              # moteur .xls (préinstallé sur Colab)
        import xlrd  # noqa: F401
    except ModuleNotFoundError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd"], check=True)
    raw = urllib.request.urlopen(SHILLER_URL, timeout=60).read()
    sheet = pd.read_excel(io.BytesIO(raw), sheet_name="Data", header=7, engine="xlrd")
    df = pd.DataFrame({
        "year":  pd.to_numeric(sheet.iloc[:, 5], errors="coerce"),          # année décimale
        "cape":  pd.to_numeric(sheet.iloc[:, 12], errors="coerce"),         # CAPE (P/E10)
        "ret10": pd.to_numeric(sheet.iloc[:, 19], errors="coerce") * 100,   # rendement réel 10 ans, %
    })
    return df.dropna(subset=["year"])

shiller = load_shiller()


from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Le futur est déjà, en grande partie, dans les cours",
        sub="Valorisation des actions américaines depuis 1881 (CAPE de Shiller)",
        ylab="CAPE (P/E ajusté du cycle)",
        mean="moyenne longue : 17,5",
        today="aujourd'hui : ~35-40\n(97ᵉ centile de l'histoire)",
        note="À plus du double de sa moyenne longue, la valorisation n'a été dépassée qu'en 1929, 2000 et 2021.\n"
             "Elle dit à quel point l'optimisme est déjà payé. Source : données de Robert Shiller."),
    "en": dict(
        title="The future is already, largely, in the price",
        sub="Valuation of U.S. equities since 1881 (Shiller CAPE)",
        ylab="CAPE (cyclically adjusted P/E)",
        mean="long-run average: 17.5",
        today="today: ~35-40\n(97th percentile of history)",
        note="At more than twice its long-run average, valuation has been exceeded only in 1929, 2000 and 2021.\n"
             "It says how much optimism is already paid for. Source: Robert Shiller's data."),
}

def build_figure(shiller: "DataFrame", lang: str) -> Figure:
    """Le CAPE de Shiller mensuel depuis 1881, sa moyenne longue et les pics de 1929/2000/2021."""
    text = LABELS[lang]
    s = shiller.dropna(subset=["cape"])
    year, cape = s["year"].to_numpy(), s["cape"].to_numpy()

    fig = nm.figure(height_px=1064)
    ax = nm.axes(fig, left=0.072)
    ax.plot(year, cape, color=nm.COLORS["blue"], linewidth=1.9)
    ax.axhline(17.5, color=nm.COLORS["text"], linestyle=(0, (7, 5)), linewidth=2.2, alpha=0.9)
    ax.set_ylim(0, 47)
    ax.set_yticks(range(0, 41, 10))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(1876, 2030)
    ax.set_xticks(range(1880, 2021, 20))
    ax.text(1891, 19.4, text["mean"], fontsize=21, fontweight="bold", color=nm.COLORS["text"])
    for yr, label in [(1929, "1929"), (2000, "2000"), (2021, "2021")]:
        peak = cape[(year >= yr - 1.3) & (year <= yr + 1.3)].max()
        ax.annotate(label, xy=(yr, peak), xytext=(0, 30), textcoords="offset points",
                    ha="center", va="bottom", fontsize=22, color=nm.COLORS["text"],
                    arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    ax.scatter([year[-1]], [cape[-1]], s=95, color=nm.COLORS["rose"], zorder=5)
    ax.annotate(text["today"], xy=(year[-1], cape[-1]), xytext=(1966, 39.2),
                ha="center", va="center", fontsize=21, fontweight="bold",
                color=nm.COLORS["rose"], linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["rose"], lw=1.8))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(shiller, LANG)